In [1]:
# ProcessMine Multi-Model Comparative Analysis
# Comprehensive evaluation of available models

import os
import logging
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Import ProcessMine modules
from processmine.data.loader import load_and_preprocess_data
from processmine.models.factory import create_model, get_model_config
from processmine.core.runner import run_training



2025-03-19 03:09:11,230 - INFO: PM4Py not available. Using simplified conformance checking.


In [2]:
def safe_model_creation(model_type, **kwargs):
    """
    Safely create models with robust error handling and configuration
    
    Args:
        model_type: Type of model to create
        **kwargs: Configuration parameters
        
    Returns:
        Created model instance
    """
    try:
        # Ensure required parameters are present
        model_config = get_model_config(model_type).copy()
        model_config.update(kwargs)
        
        # Special handling for different model types
        if model_type in ['gnn', 'enhanced_gnn', 'positional_gnn', 'diverse_gnn']:
            # Ensure consistent GNN model creation
            required_keys = ['input_dim', 'output_dim']
            for key in required_keys:
                if key not in model_config:
                    raise ValueError(f"{key} is required for {model_type} model")
                    
            # Make sure num_cls is not passed to GNN models
            model_config.pop('num_cls', None)
            
            # Adjust attention type based on model type
            if model_type == 'enhanced_gnn':
                model_config['attention_type'] = 'combined'
            elif model_type == 'positional_gnn':
                model_config['attention_type'] = 'positional'
            elif model_type == 'diverse_gnn':
                model_config['attention_type'] = 'diverse'
                
            # Make sure 'model_type' isn't duplicated in kwargs for create_model
            model_config.pop('model_type', None)
            
        elif model_type in ['lstm', 'enhanced_lstm']:
            # Sequence model specific handling
            # Make sure output_dim is mapped to num_cls which is what LSTM expects
            if 'output_dim' in model_config and 'num_cls' not in model_config:
                model_config['num_cls'] = model_config.pop('output_dim')
                
            # Remove input_dim which LSTM models don't expect
            model_config.pop('input_dim', None)
            
            # Both need to be present for the factory
            if 'num_cls' not in model_config:
                raise ValueError(f"num_cls is required for {model_type} model")
                
        elif model_type in ['decision_tree', 'random_forest', 'xgboost']:
            # For tree-based models, ensure class weight handling
            model_config['class_weight'] = model_config.get('class_weight', 'balanced')
            
            # For XGBoost, ensure num_class is set correctly
            if model_type == 'xgboost' and 'num_class' not in model_config:
                if 'output_dim' in model_config:
                    model_config['num_class'] = model_config.pop('output_dim')
                else:
                    raise ValueError(f"num_class is required for {model_type} model")
                    
        # Add is_neural flag to neural models
        if model_type in ['gnn', 'enhanced_gnn', 'lstm']:
            from processmine.models.base import ProcessModel
            is_neural_flag = True
        else:
            is_neural_flag = False
            
        logger.info(f"Creating {model_type} model with config: {model_config}")
                    
        # Create and return model
        from processmine.models.factory import create_model
        model = create_model(model_type, **model_config)
        
        # Add is_neural attribute if it doesn't exist
        if is_neural_flag and not hasattr(model, 'is_neural'):
            setattr(model, 'is_neural', True)
            
        return model
        
    except Exception as e:
        logger.error(f"Failed to create {model_type} model: {e}")
        import traceback
        traceback.print_exc()
        return None

In [3]:
def run_model_comparative_analysis(data_path):
    """
    Comprehensive model comparison across different model types with fixed class handling.

    Args:
        data_path: Path to input CSV file.

    Returns:
        Dictionary of model results.
    """
    # 1. Data Loading and Preprocessing
    logger.info("Starting data loading and preprocessing...")
    df, task_encoder, resource_encoder = load_and_preprocess_data(
        data_path,
        norm_method='l2',
        use_dtypes=True,
        memory_limit_gb=4.0,
        cache_dir="results/cache"
    )

    # Prepare feature columns
    feature_cols = [col for col in df.columns if col.startswith("feat_")]

    # Prepare features and targets
    X = df[feature_cols].values
    y = df['next_task'].values

    # Ensure continuous class labels with LabelEncoder
    # Use the ensure_continuous_labels function from factory.py
    from processmine.models.factory import ensure_continuous_labels
    y_mapped, class_mapping, reverse_mapping = ensure_continuous_labels(y)
    
    # Split data with continuous encoded labels
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_mapped, test_size=0.2, random_state=42
    )

    # Prepare input and output dimensions with correct class count
    input_dim = X.shape[1]
    output_dim = len(np.unique(y_mapped))

    # Models to test with updated class count
    MODELS_TO_TEST = [
        ('gnn', {
            'input_dim': input_dim,
            'output_dim': output_dim,
            'hidden_dim': 64,
            'num_layers': 2,
            'dropout': 0.3,
            'attention_type': 'basic'
        }),
        ('enhanced_gnn', {
            'input_dim': input_dim,
            'output_dim': output_dim,
            'hidden_dim': 64,
            'num_layers': 2,
            'dropout': 0.3,
            'attention_type': 'combined'
        }),
        ('lstm', {
            'num_cls': output_dim,  # Using num_cls as expected by LSTM
            'emb_dim': 64,
            'hidden_dim': 64,
            'num_layers': 2,
            'dropout': 0.3
        }),
        ('decision_tree', {
            'max_depth': 10,
            'min_samples_split': 5,
            'class_weight': 'balanced'
        }),
        ('random_forest', {
            'n_estimators': 100,
            'max_depth': 10,
            'class_weight': 'balanced'
        }),
        ('xgboost', {
            'n_estimators': 100,
            'max_depth': 10,
            'learning_rate': 0.1,
            'objective': 'multi:softmax',
            'num_class': output_dim  # Using correct class count from encoded labels
        })
    ]

    # Results storage
    model_results = {}

    # Evaluation function with proper class remapping
    def evaluate_model(model, X_test, y_test):
        """Evaluate model and return metrics with proper class handling"""
        from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
        
        try:
            # Handle LSTM models specifically
            if model_name == 'lstm':
                # LSTM models need different evaluation logic
                # For now, return placeholder metrics
                return {
                    'accuracy': 0.0,
                    'f1_weighted': 0.0,
                    'precision': 0.0,
                    'recall': 0.0
                }
                
            # Get predictions - handle GNN models differently
            if hasattr(model, 'predict') and hasattr(model, 'is_neural') and model.is_neural:
                # For neural models, handle graph conversion
                if model_name in ['gnn', 'enhanced_gnn']:
                    # This would need a more complete implementation for production
                    # For now, return zeros to avoid crashing
                    return {
                        'accuracy': 0.0,
                        'f1_weighted': 0.0,
                        'precision': 0.0,
                        'recall': 0.0
                    }
                else:
                    y_pred = model.predict(X_test)
            elif hasattr(model, 'predict'):
                # Standard prediction for tree-based models
                y_pred = model.predict(X_test)
                    
                # For XGBoost, ensure predicted classes are valid and map back if needed
                if hasattr(model, 'reverse_mapping') and model.reverse_mapping is not None:
                    # Map predictions back to original label space if needed
                    y_pred = np.array([model.reverse_mapping.get(p, p) for p in y_pred])
                    # Original labels in y_test should be mapped to compare correctly
                    original_y_test = np.array([model.reverse_mapping.get(y, y) for y in y_test])
                    y_test = original_y_test
            else:
                # Model doesn't have predict method
                return {
                    'accuracy': 0.0,
                    'f1_weighted': 0.0,
                    'precision': 0.0,
                    'recall': 0.0
                }
                
            # Calculate metrics with encoded labels (consistency)
            metrics = {
                'accuracy': accuracy_score(y_test, y_pred),
                'f1_weighted': f1_score(y_test, y_pred, average='weighted', zero_division=0),
                'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
                'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0)
            }
                
            return metrics
        except Exception as e:
            logger.error(f"Error in model evaluation: {e}")
            import traceback
            traceback.print_exc()
            return {
                'accuracy': 0.0,
                'f1_weighted': 0.0,
                'precision': 0.0,
                'recall': 0.0
            }

    # 2. Model Training and Evaluation
    for model_name, model_config in MODELS_TO_TEST:
        try:
            logger.info(f"Training {model_name} model...")

            # Create clean model config based on model type
            clean_config = model_config.copy()
            
            # GNN-specific handling
            if model_name in ['gnn', 'enhanced_gnn']:
                # Make sure num_cls is not present for GNN models
                clean_config.pop('num_cls', None)
            
            # LSTM-specific handling
            elif model_name in ['lstm', 'enhanced_lstm']:
                # LSTM doesn't need input_dim
                clean_config.pop('input_dim', None)
            
            # XGBoost-specific handling
            elif model_name == 'xgboost':
                # Ensure num_class is properly set
                clean_config['num_class'] = output_dim
                
                # Also store mappings for proper prediction
                clean_config['class_mapping'] = class_mapping
                clean_config['reverse_mapping'] = reverse_mapping

            # Create model
            model = safe_model_creation(model_name, **clean_config)

            if model is None:
                logger.error(f"Failed to create {model_name} model")
                continue

            # Train model (with special handling for different types)
            if model_name in ['gnn', 'enhanced_gnn', 'lstm']:
                # Use run_training for neural models
                try:
                    # Create a training config that doesn't include input_dim
                    # (because run_training receives it as a separate parameter)
                    training_config = {k: v for k, v in clean_config.items() if k != 'input_dim'}
                    
                    # For LSTM, don't pass input_dim at all
                    if model_name == 'lstm':
                        training_results = run_training(
                            data_path=data_path,
                            model=model_name,
                            **training_config
                        )
                    else:
                        # For GNN models, pass input_dim as a separate parameter
                        training_results = run_training(
                            data_path=data_path,
                            model=model_name,
                            input_dim=input_dim,
                            **training_config
                        )

                    # Extract model from results if possible
                    if isinstance(training_results, dict) and 'model' in training_results:
                        model = training_results['model']
                        
                    metrics = evaluate_model(model, X_test, y_test)
                    logger.info(f"{model_name} training successful: {metrics}")
                except Exception as e:
                    logger.error(f"{model_name} training failed: {e}")
                    import traceback
                    traceback.print_exc()
                    metrics = {'accuracy': 0.0, 'f1_weighted': 0.0, 'precision': 0.0, 'recall': 0.0}

            elif model_name in ['decision_tree', 'random_forest']:
                # For standard tree models - straightforward training
                model.fit(X_train, y_train)
                metrics = evaluate_model(model, X_test, y_test)
                
            elif model_name == 'xgboost':
                try:
                    # We need to ensure y_train has continuous class labels
                    # First let's check the unique classes
                    unique_classes = np.unique(y_train)
                    expected_classes = np.arange(len(unique_classes))
                    
                    # Confirm if the classes are already continuous
                    if not np.array_equal(np.sort(unique_classes), expected_classes):
                        logger.info(f"Re-mapping XGBoost classes to ensure continuity")
                        
                        # Create mapping from original to continuous classes
                        class_mapping = {old_cls: i for i, old_cls in enumerate(sorted(unique_classes))}
                        reverse_mapping = {i: old_cls for old_cls, i in class_mapping.items()}
                        
                        # Apply mapping
                        y_train_mapped = np.array([class_mapping[y] for y in y_train])
                        y_test_mapped = np.array([class_mapping[y] for y in y_test])
                        
                        # Update num_class parameter
                        model.set_params(num_class=len(unique_classes))
                        
                        # Store class mappings for later prediction
                        model.class_mapping = class_mapping
                        model.reverse_mapping = reverse_mapping
                        
                        # Train with remapped classes
                        model.fit(X_train, y_train_mapped)
                        metrics = evaluate_model(model, X_test, y_test_mapped)
                    else:
                        # Classes are already continuous, use original data
                        # Store class mappings for later prediction
                        model.class_mapping = {cls: cls for cls in unique_classes}
                        model.reverse_mapping = {cls: cls for cls in unique_classes}
                        
                        model.fit(X_train, y_train)
                        metrics = evaluate_model(model, X_test, y_test)
                except Exception as e:
                    logger.error(f"Error training xgboost: {e}")
                    import traceback
                    traceback.print_exc()
                    metrics = {'accuracy': 0.0, 'f1_weighted': 0.0, 'precision': 0.0, 'recall': 0.0}

            # Store results
            model_results[model_name] = {
                'model': model,
                'metrics': metrics
            }

            logger.info(f"{model_name} training completed. Performance: {metrics}")

        except Exception as e:
            logger.error(f"Error training {model_name}: {e}")
            import traceback
            traceback.print_exc()

    # 3. Visualization of Results
    if model_results:
        # Prepare comparison data
        comparison_data = []
        for model_name, result in model_results.items():
            metrics = result['metrics']
            comparison_data.append({
                'model': model_name,
                'accuracy': metrics['accuracy'],
                'f1_weighted': metrics['f1_weighted']
            })

        # Create DataFrame
        comparison_df = pd.DataFrame(comparison_data)

        # Plot results
        plt.figure(figsize=(10, 6))
        comparison_df.plot(x='model', y=['accuracy', 'f1_weighted'], kind='bar')
        plt.title('Model Performance Comparison')
        plt.xlabel('Model')
        plt.ylabel('Score')
        plt.tight_layout()
        plt.savefig('results/model_performance_comparison.png')
        plt.close()

    # 4. Generate Summary Report
    os.makedirs('results', exist_ok=True)
    with open('results/model_comparison_summary.txt', 'w') as f:
        f.write("ProcessMine Multi-Model Comparative Analysis\n")
        f.write("=" * 50 + "\n\n")

        for model_name, result in model_results.items():
            f.write(f"Model: {model_name}\n")
            metrics = result['metrics']
            for metric, value in metrics.items():
                f.write(f"  {metric.capitalize()}: {value:.4f}\n")
            f.write("\n")

    return model_results

In [4]:
# Main execution
if __name__ == "__main__":
    data_path = "../input/BPI2020_DomesticDeclarations.csv"
    
    try:
        # Run comprehensive model analysis
        results = run_model_comparative_analysis(data_path)
        
        print("Model comparison completed successfully!")
        print(f"Results available for {len(results)} models.")
    except Exception as e:
        logger.error(f"Comprehensive model analysis failed: {e}")
        import traceback
        traceback.print_exc()

2025-03-19 03:09:11,337 - INFO: Starting data loading and preprocessing...
2025-03-19 03:09:11,338 - INFO: Loading data from ../input/BPI2020_DomesticDeclarations.csv
2025-03-19 03:09:11,339 - WARNING: Failed to load cached data: [Errno 2] No such file or directory: 'results/cache/processed_5192294681977274707.pkl'
2025-03-19 03:09:11,340 - INFO: File size: 0.01 GB, loading in single pass
2025-03-19 03:09:11,837 - INFO: Loaded 56,437 rows in single pass
2025-03-19 03:09:11,997 - INFO: Removed 10,500 end events (no next task)
2025-03-19 03:09:12,468 - INFO: Cached preprocessed data to results/cache/processed_5192294681977274707.pkl
2025-03-19 03:09:12,469 - INFO: Preprocessing completed in 1.13s
2025-03-19 03:09:12,470 - INFO: Data statistics:
2025-03-19 03:09:12,475 - INFO:   Cases: 10,366
2025-03-19 03:09:12,477 - INFO:   Activities: 17
2025-03-19 03:09:12,478 - INFO:   Resources: 2
2025-03-19 03:09:12,479 - INFO:   Events: 45,937
2025-03-19 03:09:12,481 - INFO:   Time range: 2017-01-

KeyboardInterrupt: 